# Step 11 — Summarize evaluation (single-occurrence filter + IRR)

> Two jobs over the manually filled `11_manual_evaluation.csv`:
> 1. **Filter** — write a single-occurrence variant that drops every multi-occurrence present row (`final_decision == "present"` and `final_count > 1`); absent and single-occurrence present rows are kept verbatim. This is the CSV every downstream step consumes.
> 2. **IRR** — compute Cohen's κ per annotator pair and overall (mean ± std) plus the Landis-Koch band, over the **filtered** rows.

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/11_manual_evaluation.csv` | CSV — one row per (file_key, target_smell): paired columns `annotator_a`/`decision_a`, `annotator_b`/`decision_b`, plus `disagreement`, adjudicated `final_decision`, and `final_count` | Manual — produced offline by annotator pairs from `data/10_ground_truth_sample.json` |

## Outputs
| Path | Format | Consumer |
|---|---|---|
| `data/11_manual_evaluation_with_one_occurrence_by_smell.csv` | CSV (raw schema, multi-occurrence present rows removed) | Step 12 (ground_truth build) |
| `data/11_irr_results.csv` | CSV (per-pair κ + concordance + Landis-Koch band, on filtered rows) | Step 11b (LaTeX tables) |
| `data/step11_report.json` | JSON (filter stats + per-pair confusion matrices) | Step 11b |

## Manual intervention
Requires `data/11_manual_evaluation.csv` produced offline by annotators — not shipped automatically. The current input has 8 annotators in 4 pairs (A1-A4, A2-A3, A5-A8, A6-A7).


In [1]:
# --- Notebook bootstrap ---
# Idempotent: chdir to repo root + reset sys.argv for argparse.
# Works in Jupyter/JupyterLab/Antigravity/VS Code/plain python on Linux+macOS+Windows.
from dataset_utils.notebook_bootstrap import setup

setup()

print("Step 11 — starting: summarize evaluation (single-occurrence filter + IRR)...", flush=True)
"""Step 11: Summarize the manual evaluation.

Two jobs, run in order:

  1. Filter — read the raw manual evaluation and write a single-occurrence
     variant. Drop every row whose consensus is a multi-occurrence present
     (final_decision == "present" and final_count > 1). Absent rows
     (final_count == 0) and single-occurrence present rows are kept verbatim
     (same header, same column order). This single-occurrence CSV is the input
     consumed by all downstream steps (12 onward).

  2. IRR — compute Cohen's Kappa per annotator pair, mean Kappa (± std),
     concordance, and annotator A-vs-B confusion matrices over the FILTERED
     rows (decisions come from decision_a / decision_b).

The filter + IRR math lives in dataset_utils.irr (ONE implementation, shared with
Step 12e so the two can't drift); this notebook only drives it.

Inputs:
  data/11_manual_evaluation.csv

Outputs:
  data/11_manual_evaluation_with_one_occurrence_by_smell.csv
                                   Single-occurrence variant (downstream input)
  data/11_irr_results.csv          Per-pair and overall IRR metrics (on filtered rows)
  data/step11_report.json          Full report: filter stats + per-pair confusion matrices
"""

import argparse
import json
import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.hashing import compute_file_hash
from dataset_utils.irr import compute_irr, filter_one_occurrence, print_summary, write_irr_csv

MANUAL_EVAL_PATH = Path("data/11_manual_evaluation.csv")
FILTERED_PATH = Path("data/11_manual_evaluation_with_one_occurrence_by_smell.csv")
OUTPUT_CSV_PATH = Path("data/11_irr_results.csv")
REPORT_PATH = Path("data/step11_report.json")

def run(
    eval_path: Path = MANUAL_EVAL_PATH,
    filtered_path: Path = FILTERED_PATH,
    output_csv: Path = OUTPUT_CSV_PATH,
    report_path: Path = REPORT_PATH,
) -> dict:
    # 1. Filter the raw evaluation -> single-occurrence CSV (downstream input).
    filter_stats = filter_one_occurrence(eval_path, filtered_path)

    # 2. IRR over the filtered rows.
    result = compute_irr(filtered_path)
    write_irr_csv(result, output_csv)

    report = {
        **result,
        "filter": filter_stats,
        "input_sha256": compute_file_hash(eval_path),
        "filtered_file": str(filtered_path),
        "filtered_sha256": compute_file_hash(filtered_path),
        "output_file": str(output_csv),
        "output_sha256": compute_file_hash(output_csv),
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return report

def _parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Summarize manual evaluation: single-occurrence filter + IRR."
    )
    p.add_argument("--input", type=Path, default=MANUAL_EVAL_PATH)
    p.add_argument("--filtered", type=Path, default=FILTERED_PATH)
    p.add_argument("--output-csv", type=Path, default=OUTPUT_CSV_PATH)
    p.add_argument("--report", type=Path, default=REPORT_PATH)
    return p.parse_args()

def main() -> None:
    args = _parse_args()
    report = run(args.input, args.filtered, args.output_csv, args.report)
    fs = report["filter"]
    print("Summarize evaluation complete.")
    print(f"  Input        : {args.input}")
    print(f"  Filtered CSV : {args.filtered}")
    print(f"  IRR CSV      : {args.output_csv}")
    print(f"  Report       : {args.report}")
    print(
        f"  Filter       : {fs['n_in']} -> {fs['n_out']} rows"
        f"  ({fs['n_dropped']} multi-occurrence present dropped)"
    )
    print_summary(report)

main()

Step 11 — starting: summarize evaluation (single-occurrence filter + IRR)...
Summarize evaluation complete.
  Input        : data/11_manual_evaluation.csv
  Filtered CSV : data/11_manual_evaluation_with_one_occurrence_by_smell.csv
  IRR CSV      : data/11_irr_results.csv
  Report       : data/step11_report.json
  Filter       : 1846 -> 1769 rows  (77 multi-occurrence present dropped)

  Pair             N   Kappa Interpretation      Concordance
  ------------ ----- ------- ------------------ ------------
  A1-A4          710  0.7670 Substantial              88.45%
  A2-A3           30  0.5200 Moderate                 86.67%
  A5-A8          438  0.7853 Substantial              89.27%
  A6-A7          591  0.9492 Almost perfect           97.46%
  ------------ ----- ------- ------------------ ------------
  mean          1769  0.7554 ±0.1770                 91.63%



---

**Next step:** [`11b_latex_tables.ipynb`](./11b_latex_tables.ipynb)
